# Introduction to `matplotlib`

This notebook covers Sect. III of [02_01_Modules_in_python_Overview.ipynb](02_01_Modules_in_python_Overview.ipynb). 

The goal is *not* to learn every option of matplotlib: the [gallery](https://matplotlib.org/stable/gallery/index) and AI assistants will happily give you the code for almost any figure. The goal is for you to be able to draw basic figures by yourself but also to understand the **core logic** well enough to read, adapt and debug that code.

## Table of Contents

| Section | Content | 
|---|---|
| [III.1 A first plot](#III.1) | `plt.plot`, labels, legend, line styles | 
| [III.2 Figures and axes: `plt.subplots()`](#III.2) | the object-oriented interface, multiple panels, saving | 
| [III.3 Overview of key plot types](#III.3) | scatter, errorbar, histogram, log scale | 
| [III.4 Images and functions on a grid](#III.4) | `imshow`, `np.meshgrid`, `contour`, colorbars | 
| [III.5 Going further](#III.5) | good practices, gallery, AI, references | 

## III. `matplotlib`: Visualisation with Python <a class="anchor" id="III"></a>

`matplotlib` is the standard plotting library of the scientific Python ecosystem. It produces publication-quality figures in many formats (png, pdf, svg, ...) and many other libraries (`pandas`, `seaborn`, `astropy`, ...) are built on top of it.

We almost always import its `pyplot` sub-module with the alias `plt`:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

> In recent versions of Jupyter, figures are shown inline automatically. With older setups you may need the magic command `%matplotlib inline` (or `%matplotlib widget` for interactive figures). In a plain Python script, call `plt.show()` to open the plotting window.

### III.1 A first plot <a class="anchor" id="III.1"></a>

Plotting $y$ as a function of $x$ is a single call to `plt.plot(x, y)`. A figure without axis labels is useless, so we always add them:

In [ ]:
x = np.linspace(0, 10, 100)
y = np.sin(x)

plt.plot(x, y)
plt.xlabel('x')
plt.ylabel('sin(x)')

Each call to `plt.plot()` adds a curve to the current figure. The appearance of a curve is controlled with **keyword arguments**. The most useful ones are:

| keyword | meaning | examples |
|---|---|---|
| `color` (or `c`) | line colour | `'red'`, `'C0'`, `'k'`, `'#1f77b4'` |
| `ls` (linestyle) | line style | `'-'`, `'--'`, `':'`, `'-.'`, `''` |
| `marker` | point symbol | `'o'`, `'s'`, `'x'`, `'.'` |
| `lw` (linewidth) | line width | `2` |
| `alpha` | transparency (0 → 1) | `0.5` |
| `label` | name used in the legend | `'data'` |

A shortcut *format string* combines marker, line style and colour: `plt.plot(x, y, 'o--r')`.

Add a legend with `plt.legend()`, a title with `plt.title()`, and change the plotted range with `plt.xlim()` / `plt.ylim()`. Strings starting with `r` and containing `$...$` are rendered as LaTeX math.

In [ ]:
plt.plot(x, np.sin(x), color='C0', lw=2, label=r'$\sin(x)$')
plt.plot(x, np.cos(x), color='C1', ls='--', label=r'$\cos(x)$')
plt.plot(x[::5], np.sin(x[::5]), 'o', color='C0', alpha=0.5)   # markers only, every 5th point

plt.xlabel('x')
plt.ylabel('y')
plt.title('My first figure')
plt.xlim(0, 2 * np.pi)
plt.legend()

**Exercise 1** 

Plot $y = x^2$ and $y = x^3$ for $x \in [-2, 2]$ on the same figure, with different line styles, a legend, axis labels, and restrict the $y$ range to $[-4, 4]$.

In [ ]:
# Your code here

### III.2 Figures and axes: `plt.subplots()` <a class="anchor" id="III.2"></a>

The `plt.xxx()` commands above act on the *current* figure behind the scenes. This is convenient for quick looks, but as soon as a figure gets a bit more complex it is much clearer to work with explicit objects:

- a **`Figure`** (`fig`): the whole window / page,
- one or several **`Axes`** (`ax`): the panels in which data are drawn (each with its own x and y axis).

Both are created in one line with `plt.subplots()`. This **object-oriented** style is the one used in the [matplotlib gallery](https://matplotlib.org/stable/gallery/index), in most online examples, and in the code AI assistants generate — so it is the one worth learning.

<img src="https://matplotlib.org/stable/_images/anatomy.png" width="450">

In [ ]:
x = np.linspace(0, 10, 100)

fig, ax = plt.subplots(figsize=(6, 4))      # figsize = (width, height) in inches
ax.plot(x, np.sin(x), label='sin')
ax.plot(x, np.cos(x), label='cos')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Same figure, object-oriented style')
ax.legend()

Most `plt.` functions have an `ax.set_` equivalent:

``` python
plt.xlabel()  ->  ax.set_xlabel()
plt.xlim()    ->  ax.set_xlim()
plt.title()   ->  ax.set_title()
plt.xticks()  ->  ax.set_xticks()   (+ ax.set_xticklabels())
plt.xscale()  ->  ax.set_xscale()
```
The plotting functions themselves keep the same name: `ax.plot()`, `ax.scatter()`, `ax.hist()`, `ax.imshow()`, ... Use `ax.set(xlabel=..., ylabel=..., xlim=...)` to set several properties at once. A useful (less known) method: `ax.tick_params(labelsize=12)` changes the size of the tick labels.

#### Multiple panels

`plt.subplots(nrows, ncols)` returns one figure and an **array** of axes:

``` python
fig, ax = plt.subplots(1, 1)   # ax is a single Axes
fig, ax = plt.subplots(1, 3)   # ax is a 1D array: ax[0], ax[1], ax[2]
fig, ax = plt.subplots(2, 2)   # ax is a 2D array: ax[0, 0], ax[0, 1], ...
```

Useful options: `sharex=True` / `sharey=True` (common axis range), `figsize=(w, h)`, and `layout='constrained'` which automatically avoids overlapping labels (the older equivalent is calling `fig.tight_layout()`).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5), sharex=True, layout='constrained')

for i, ax in enumerate(axes.flat):          # .flat loops over all panels
    k = i + 1
    ax.plot(x, np.sin(k * x), color=f'C{i}')
    ax.set_title(f'sin({k}x)')

for ax in axes[-1, :]:                      # x label only on the bottom row
    ax.set_xlabel('x')

#### Saving a figure

``` python
fig.savefig('myfigure.pdf')              # format deduced from the extension
fig.savefig('myfigure.png', dpi=200)     # raster format: choose the resolution
```
Use vector formats (`pdf`, `svg`) for papers and `png` for slides / web. Always save from code (not by clicking in a window) so that the figure can be regenerated identically.

**Exercise 2** 

- Create a figure with 2 panels on top of each other sharing the x axis.
- Plot $\sin(x)$ in red in the top panel and $\cos(x)$ in blue in the bottom one, for $x \in [-2\pi, 2\pi]$.
- Add a horizontal dashed line at $y=0$ in both panels (hint: `ax.axhline()`).
- *Bonus*: place x ticks at multiples of $\pi$ and label them with the $\pi$ symbol.
- Save the figure as `sincos.pdf`.

In [ ]:
# Your code here

### III.3 Overview of key plot types <a class="anchor" id="III.3"></a>

Beyond `plot`, a handful of functions cover most everyday needs. They all work the same way (`ax.function(data, **options)`):

| function | use |
|---|---|
| `ax.scatter(x, y, s=, c=)` | points with individual size `s` and colour `c` |
| `ax.errorbar(x, y, yerr=, xerr=)` | data with uncertainties |
| `ax.hist(data, bins=)` | histogram (computes and draws it) |
| `ax.set_xscale('log')` / `ax.loglog()` | logarithmic axes |
| `ax.fill_between(x, y1, y2)` | shaded band (e.g. confidence interval) |
| `ax.bar(x, height)` / `ax.step(x, y)` | bar chart / step function |

Read the cell below rather than memorising it — the point is to recognise the pattern.

In [ ]:
rng = np.random.default_rng(42)
fig, ax = plt.subplots(2, 2, figsize=(9, 7), layout='constrained')

# Scatter: colour encodes a third quantity
xs, ys = rng.normal(size=(2, 200))
sc = ax[0, 0].scatter(xs, ys, c=np.hypot(xs, ys), s=15, cmap='viridis')
fig.colorbar(sc, ax=ax[0, 0], label='distance to origin')
ax[0, 0].set_title('scatter')

# Error bars + model + shaded band
xd = np.linspace(0, 10, 12)
yd = 2 * xd + 1 + rng.normal(0, 2, xd.size)
ax[0, 1].errorbar(xd, yd, yerr=2, fmt='o', capsize=3, label='data')
ax[0, 1].plot(xd, 2 * xd + 1, 'k-', label='model')
ax[0, 1].fill_between(xd, 2 * xd - 1, 2 * xd + 3, color='grey', alpha=0.3)
ax[0, 1].legend()
ax[0, 1].set_title('errorbar + fill_between')

# Histogram
ax[1, 0].hist(rng.normal(size=5000), bins=40, histtype='stepfilled', alpha=0.6)
ax[1, 0].set_title('hist')

# Log scale
xl = np.logspace(-1, 3, 50)
ax[1, 1].loglog(xl, xl**-2, label=r'$x^{-2}$')
ax[1, 1].loglog(xl, xl**0.5, label=r'$x^{1/2}$')
ax[1, 1].legend()
ax[1, 1].set_title('log-log')

### III.4 Images and functions on a grid <a class="anchor" id="III.4"></a>

#### Displaying a 2D array: `imshow`

Any 2D array (an image, a detector frame, a map, a matrix) can be shown with `ax.imshow()`. Three options matter most:

- `origin='lower'`: by default, element `[0, 0]` is drawn at the **top-left** (image convention). Use `origin='lower'` to put it at the bottom-left (plot convention).
- `extent=[xmin, xmax, ymin, ymax]`: physical coordinates of the image edges (otherwise axes are in pixel indices).
- `cmap=` the colour map (`'viridis'`, `'gray'`, `'magma'`, `'RdBu_r'`, ...), with `vmin`/`vmax` to set the colour range.

Always add a colour bar with `fig.colorbar()` so that colours can be read as numbers.

In [ ]:
image = rng.poisson(5, size=(60, 80)).astype(float)   # a fake noisy "detector" image
image[20:30, 40:55] += 20                             # with a bright rectangle

fig, ax = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
ax[0].imshow(image)
ax[0].set_title('default (origin upper)')
im = ax[1].imshow(image, origin='lower', cmap='magma', vmin=0, vmax=30)
ax[1].set_title("origin='lower', cmap='magma'")
fig.colorbar(im, ax=ax[1], label='counts')

> To display a picture file (png, jpg), read it with `plt.imread('file.png')` and pass the resulting array to `ax.imshow()`. For astronomical FITS images, use `astropy.io.fits` to get the array.

#### Evaluating a function on a grid: `np.meshgrid`

To evaluate a function $f(x, y)$ on a 2D grid we need, for every pixel, its $x$ **and** its $y$ coordinate. `np.meshgrid` builds these two coordinate arrays from two 1D arrays:

``` python
x = np.linspace(-5, 5, m)     # m values along the x axis (columns)
y = np.linspace(-3, 3, n)     # n values along the y axis (rows)
X, Y = np.meshgrid(x, y)      # X and Y both have shape (n, m)
Z = f(X, Y)                   # numpy broadcasting does the rest
```
`X[i, j]` is the $x$ coordinate of pixel $(i, j)$, and `Y[i, j]` its $y$ coordinate. Let's look at a tiny example:

In [ ]:
X, Y = np.meshgrid(np.arange(4), np.arange(3))
print('X =\n', X)
print('Y =\n', Y)

The resulting `Z` can be shown with `imshow` (as an image) or with `contour` / `contourf` (iso-lines / filled iso-levels), which take the coordinate arrays directly:

In [ ]:
x = np.linspace(-3, 3, 200)
y = np.linspace(-2, 2, 150)
X, Y = np.meshgrid(x, y)
Z = np.sin(X**2 + Y**2) * np.exp(-0.3 * (X**2 + Y**2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')

im = ax[0].imshow(Z, origin='lower', extent=[x.min(), x.max(), y.min(), y.max()], cmap='RdBu_r')
fig.colorbar(im, ax=ax[0])
ax[0].set_title('imshow')

cf = ax[1].contourf(X, Y, Z, levels=15, cmap='RdBu_r')
cs = ax[1].contour(X, Y, Z, levels=[0], colors='k')      # iso-line Z = 0
ax[1].clabel(cs)
fig.colorbar(cf, ax=ax[1])
ax[1].set_title('contourf + contour')

for a in ax:
    a.set(xlabel='x', ylabel='y', aspect='equal')

**Exercise 3** 

1. Evaluate a 2D Gaussian $Z = \exp\left(-\frac{(x - x_0)^2 + (y - y_0)^2}{2\sigma^2}\right)$ with $\sigma = 1$, centred on $(x_0, y_0) = (0, 0)$, on a $50 \times 50$ grid covering $[-5, 5] \times [-5, 5]$.
2. In a figure with two panels, show it with `imshow` (with correct `extent`, `origin` and a colour bar) and with `contour` (+ `clabel`).
3. Move the centre to $(x_0, y_0) = (2, 0)$ and check that the Gaussian moves to the right in **both** panels. What happens in the `imshow` panel if you remove `origin='lower'` and use $(x_0, y_0) = (0, 2)$?

In [ ]:
# Your code here

### III.5 Going further <a class="anchor" id="III.5"></a>

#### Checklist before using a figure in a report or paper

- Axis labels **with units**, and a legend or colour bar when needed.
- Font sizes readable at the final size: set `figsize` to (roughly) the size the figure will have on the page, then adjust `fontsize=` / `ax.tick_params(labelsize=)`.
- Readable in black & white and by colour-blind readers: combine colour with line styles / markers, avoid red + green together, prefer perceptually uniform colour maps (`viridis`, `magma`, `cividis`) over `jet`.
- Vector format (`pdf`) for papers, `png` with `dpi≥150` for slides.
- Consistent look across figures: set defaults once with `plt.rcParams[...]` or a style file (`plt.style.use('style_name')`, see `plt.style.available`).

#### Using the gallery and AI assistants efficiently

- Browse the [gallery](https://matplotlib.org/stable/gallery/index) to find a figure close to the one you want and start from its code.
- When asking an AI assistant, describe the figure precisely (panels, what goes on each axis, style) and ask for the **object-oriented** (`fig, ax`) style. You now know enough to read the result, check it, and modify it — don't use code you don't understand.
- Twin axes, insets, irregular panel layouts (`GridSpec`, `subplot_mosaic`), rc parameters and style files are covered in [Modules_in__python_matplotlib_adv.ipynb](Modules_in__python_matplotlib_adv.ipynb).

#### References

- [Matplotlib gallery](https://matplotlib.org/stable/gallery/index): hundreds of examples with source code — the most useful resource.
- [Quick start guide](https://matplotlib.org/stable/users/explain/quick_start.html): the official introduction to figures and axes.
- [Cheat sheets](https://matplotlib.org/cheatsheets/): one-page summaries of the main functions and options.
- [Choosing colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html)
- [Seaborn](https://seaborn.pydata.org/): statistical visualisation built on matplotlib.

### Summary

- `import matplotlib.pyplot as plt`; quick plots with `plt.plot(x, y)` + labels and legend.
- For anything beyond a quick look: `fig, ax = plt.subplots(nrows, ncols)`, then `ax.plot(...)`, `ax.set_xlabel(...)`, ... and `fig.savefig(...)`.
- Other plot types (`scatter`, `errorbar`, `hist`, log axes, ...) follow the same pattern.
- 2D arrays are shown with `ax.imshow()` (mind `origin` and `extent`) + `fig.colorbar()`; functions of two variables are evaluated with `np.meshgrid` and shown with `imshow` or `contour(f)`.